# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Greemines/Flyrank-Notebook-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## **My Rule**

I want to identify pages that receive search visibility but have a relatively low click-through rate. These pages represent opportunities where improving titles, meta descriptions, or search snippets may increase traffic without needing additional impressions.

My baseline score is based on three signals:

- Higher impressions increase the score because there is enough search visibility.
- Lower CTR increases the score because there is greater room for improvement.
- Better search position increases the score because pages already appearing on the first page have the best chance of gaining additional clicks.

The page with the highest score becomes the highest-priority optimization candidate.

### **Reason Code**

CTR_OPPORTUNITY

### **Action Label**

Review title, meta description, and search snippet to improve click-through rate.

In [ ]:
import pandas as pd

df = pd.read_csv("/content_refresh_anonymized.csv")

df[[
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position"
]].head()


,content_id,impressions_90d,ctr,avg_position
0,content_304f48230142,3803,0.76,10.6
1,content_a1fb4e703a9e,15320,0.05,20.3
2,content_9aa793d4d895,12581,0.09,36.5
3,content_331d6c4de07b,11751,0.49,6.2
4,content_d99b7a2d90ca,19140,0.13,44.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Build the Ranked Queue

This baseline score prioritizes pages with high search visibility but low click-through rates. The score combines impressions, CTR, and search position into a simple hand-written rule.

Each page receives:
- A baseline score
- One reason code
- One recommended action

The ranked queue is written into `work/outputs/baseline_action_score.csv.`
(The file was added via upload)

In [ ]:
import pandas as pd
import numpy as np
import os


df = pd.read_csv("/content_refresh_anonymized.csv")


df = df[df["impressions_90d"] >= 100].copy()

df["impression_score"] = (
    np.log1p(df["impressions_90d"])
    / np.log1p(df["impressions_90d"].max())
)

# -------------------------
# Lower CTR = higher opportunity
# -------------------------

df["ctr_score"] = (
    1 - (df["ctr"] / df["ctr"].max())
).clip(0, 1)

# -------------------------
# Better ranking = higher opportunity
# -------------------------

max_position = df["avg_position"].max()

df["position_score"] = (
    1 - (df["avg_position"] / max_position)
).clip(0, 1)

# -------------------------
# Final Baseline Score
# -------------------------

df["baseline_score"] = (
    0.50 * df["impression_score"] +
    0.30 * df["ctr_score"] +
    0.20 * df["position_score"]
)

# -------------------------
# Reason Code
# -------------------------

df["reason_code"] = "CTR_OPPORTUNITY"

# -------------------------
# Action
# -------------------------

df["action"] = (
    "Review title, meta description, and search snippet"
)

# -------------------------
# Rank Pages
# -------------------------

df = df.sort_values(
    by="baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# -------------------------
# Save CSV
# -------------------------

os.makedirs("work/outputs", exist_ok=True)

output = df[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "ctr",
        "avg_position",
    ]
]

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("✅ baseline_action_score.csv written successfully!")

# -------------------------
# Preview Top 20
# -------------------------

output.head(20)

✅ baseline_action_score.csv written successfully!


,rank,content_id,baseline_score,reason_code,action,impressions_90d,ctr,avg_position
0,1,content_8c19996aa890,0.989923,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",509252,0.15,2.5
1,2,content_5fe46e04994d,0.986980,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",517715,0.14,4.2
2,3,content_aaef01a50def,0.981429,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",517109,0.25,5.4
3,4,content_4c36c775b818,0.980130,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",463103,0.41,2.3
4,5,content_1a9e894be2e2,0.976838,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",416180,0.23,4.0
5,6,content_8451fc6f034d,0.969622,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",272144,0.03,2.3
6,7,content_db5989a78dd3,0.967082,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",345111,0.21,5.4
7,8,content_cb112fce36be,0.963820,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",309910,0.16,5.6
8,9,content_2c2606c5d176,0.961870,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",347399,0.53,4.2
9,10,content_44e481c8f55b,0.961108,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",312694,0.65,1.4


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-20 Review**

The rule selected pages with high visibility, relatively low CTR, and good search positions as the best candidates for CTR optimization.

Most of the top-ranked pages have hundreds of thousands of impressions, which increases confidence that improving their title or meta description could affect a meaningful number of users. Confidence is lower for pages with fewer impressions because any improvement would have a smaller overall impact.

The recommendation could be wrong if a page already has a high CTR, if poor ranking (*rather than the search snippet*) is limiting clicks, or if the observed CTR is mainly influenced by search intent or strong competing results instead of the page's title or meta description.

In [ ]:
top20 = output.head(20).copy()

reviews = []

for _, row in top20.iterrows():

    # Confidence
    if row["impressions_90d"] > 400000:
        confidence = "High"
    elif row["impressions_90d"] > 250000:
        confidence = "Medium"
    else:
        confidence = "Low"

    # What could make it wrong?
    if row["ctr"] > 0.80:
        wrong = "CTR is already high, so improving the snippet may have limited impact."
    elif row["avg_position"] > 10:
        wrong = "Poor ranking may be limiting clicks more than the title or meta description."
    else:
        wrong = "The page may already be optimized, so the low CTR could be caused by user intent or strong competitors."

    reviews.append({
        "Rank": row["rank"],
        "Content ID": row["content_id"],
        "Action": row["action"],
        "Reason Code": row["reason_code"],
        "Confidence": confidence,
        "What would make it wrong": wrong
    })

review_df = pd.DataFrame(reviews)

review_df


,Rank,Content ID,Action,Reason Code,Confidence,What would make it wrong
0,1,content_8c19996aa890,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,High,"The page may already be optimized, so the low ..."
1,2,content_5fe46e04994d,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,High,"The page may already be optimized, so the low ..."
2,3,content_aaef01a50def,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,High,"The page may already be optimized, so the low ..."
3,4,content_4c36c775b818,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,High,"The page may already be optimized, so the low ..."
4,5,content_1a9e894be2e2,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,High,"The page may already be optimized, so the low ..."
5,6,content_8451fc6f034d,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,Medium,"The page may already be optimized, so the low ..."
6,7,content_db5989a78dd3,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,Medium,"The page may already be optimized, so the low ..."
7,8,content_cb112fce36be,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,Medium,"The page may already be optimized, so the low ..."
8,9,content_2c2606c5d176,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,Medium,"The page may already be optimized, so the low ..."
9,10,content_44e481c8f55b,"Review title, meta description, and search sni...",CTR_OPPORTUNITY,Medium,"The page may already be optimized, so the low ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# Weak picks + leakage check

A few of the selected pages may be weak recommendations because they already have relatively high CTR values, meaning there may be limited room for improvement through title or meta description changes. Likewise, pages with lower impressions have less potential business impact even if their CTR improves.

No leakage was introduced into this baseline. The rule only uses impressions_90d, ctr, and avg_position, which are observable metrics rather than future outcomes or product-generated flags. No future-window features, labels, or decision flags were used to calculate the score.

In [ ]:
# Pages that may be weak picks
weak_picks = output[
    (output["ctr"] > 0.80) |
    (output["impressions_90d"] < 500)
]

print("Number of weak picks:", len(weak_picks))

weak_picks

Number of weak picks: 6264


,rank,content_id,baseline_score,reason_code,action,impressions_90d,ctr,avg_position
12,13,content_9532f197bbc8,0.953718,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",309192,0.87,2.0
26,27,content_8e7ba84a972b,0.943501,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",288426,0.92,4.8
27,28,content_89e84d699e9e,0.942486,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",275226,0.89,4.8
86,87,content_03d2673b2553,0.925743,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",143314,0.83,1.9
112,113,content_ba2acb4ebd04,0.921588,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",142072,0.83,3.6
...,...,...,...,...,...,...,...,...
22001,22002,content_7bba7f840807,0.484585,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",111,7.21,4.8
22002,22003,content_42b545c5e03b,0.448622,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",106,8.49,5.5
22003,22004,content_f2f27ac7fcdd,0.399876,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",138,10.87,4.6
22004,22005,content_c6a9f1c16dee,0.391526,CTR_OPPORTUNITY,"Review title, meta description, and search sni...",180,11.67,3.7


In [ ]:
# Leakage test
# Features actually used by the baseline rule
features_used = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

# Columns that would leak future information or labels
leakage_columns = [
    "trend_direction",
    "trend_pct"
]

print("Features used:", features_used)
print("Leakage columns in dataset:", leakage_columns)

used_leakage = [c for c in leakage_columns if c in features_used]

if len(used_leakage) == 0:
    print("✅ PASS: No leakage columns were used in the baseline rule.")
else:
    print("❌ FAIL: Leakage detected:", used_leakage)

Features used: ['impressions_90d', 'ctr', 'avg_position']
Leakage columns in dataset: ['trend_direction', 'trend_pct']
✅ PASS: No leakage columns were used in the baseline rule.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.